# Riffle Baseline Generation Analysis

In [ ]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 37.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os, subprocess
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

# System dependencies
!apt-get install -y fluidsynth

is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
_vllm = 'vllm==0.9.2' if is_t4 else 'vllm==0.15.1'
_triton = 'triton==3.2.0' if is_t4 else 'triton'

!pip install uv
!uv pip install -qqq --upgrade {_vllm} bitsandbytes xformers unsloth
!uv pip install -qqq {_triton}
!uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install music21 midiutil midi2audio pydub datasets --quiet
!uv pip install "numpy<2.0" # needed???

from google.colab import drive
drive.mount('/content/drive')
import sys
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

if not os.path.exists("/content/riffle"):
    !git clone --branch dev https://github.com/ronaldleung1/riffle.git
else:
    !cd /content/riffle && git pull origin unsloth

sys.path.insert(0, "/content/riffle")
os.chdir("/content/riffle")
print("Ready.")

In [ ]:
# UNCOMMENT AND RUN BELOW TO UPDATE

!cd riffle && git pull

import importlib
import chord_rewards, baseline_chord_gen
importlib.reload(chord_rewards)
importlib.reload(baseline_chord_gen)
from baseline_chord_gen import generate_sectional

/bin/bash: line 1: cd: riffle: No such file or directory


In [ ]:
from baseline_chord_gen import generate_sectional

result = generate_sectional(
    style="jazz",
    sections=["intro", "verse", "chorus", "verse", "chorus", "outro"],
    output_mp3_path="/content/base_jazz.mp3",
)
result.play()

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

full base-model eval (saves results to Drive):

In [ ]:
import os; os.chdir("/content/riffle")

!python eval.py \
    --base-model unsloth/Qwen3-1.7B \
    --samples 5 \
    --out-dir /content/drive/MyDrive/riffle_eval/base

# **[ARCHIVE]**

## Generate a single chord progression (baseline)

In [ ]:
from baseline_chord_gen import generate
result = generate(
    key="D", mode="minor", style="jazz", num_bars=8,
    output_midi_path="output.mid",
    output_mp3_path="output.mp3",
    output_report_path="output.txt",
)
print(result)

## Generate a batch of chord progressions

In [ ]:
from baseline_chord_gen import generate_batch

batch = generate_batch(
    n=10,
    key="D", mode="minor", style="jazz", num_bars=8,
    temperature=0.8,
    save_best_midi="best.mid",
    save_best_mp3="best.mp3",
)

# batch.results          # list of GenerationResult
# batch.mean_reward      # aggregate stats also: pass_rate, std_reward, median_reward, min/max
# batch.mean_breakdown   # per-component means: key / style / voice
# batch.best()           # highest-reward run

## Generate a variety of chord progressions

In [ ]:
from baseline_chord_gen import generate_grid

grid = generate_grid(
    keys=["C", "D", "F"],
    modes=["major", "minor"],
    styles=["jazz", "pop", "blues"],
    num_bars=[8],
    samples_per_cell=5,
    temperature=0.8,
)

# Tabular view (pandas optional)
import pandas as pd
df = pd.DataFrame(grid.as_rows())
df

# grid.group_by("style")    # collapse across keys/modes — which style is hardest?
# grid.group_by("key")
# grid.best()               # single best run across the whole sweep
# grid.overall_mean_reward, grid.overall_mean_breakdown